In [1]:
needed_cols = ["PWGTP", "AGEP", "SEX", "RAC1P", "HISP", "SCHL", "PINCP", "PUMA"]

In [3]:
import pandas as pd

file_path = r"C:\Users\kband\Documents\ACS2020-2021\psam_pusa.csv"

df_a = pd.read_csv(
    file_path,
    usecols=needed_cols,
    low_memory=False,
    chunksize=100000
)

In [5]:
chunks = []

for chunk in pd.read_csv(
    file_path,
    usecols=needed_cols,
    low_memory=False,
    chunksize=100000
):
    chunks.append(chunk)

df_a = pd.concat(chunks, ignore_index=True)

In [7]:
test = pd.read_csv(
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusa.csv",
    usecols=needed_cols,
    nrows=5
)
print(test.head())

   PUMA  PWGTP  AGEP  SCHL  SEX  HISP  PINCP  RAC1P
0   700     21    84    19    2     1  22510      1
1   700     19    84    21    1     1  14100      1
2   900     19    78    19    2     1  45800      1
3   302     34    46    20    1     1  65000      1
4   302     30    52    18    2     1      0      1


In [9]:
import pandas as pd
import numpy as np

file_path = r"C:\Users\kband\Documents\ACS2020-2021\psam_pusa.csv"

usecols = ["PWGTP", "AGEP", "SEX", "HISP", "RAC1P", "SCHL", "PINCP"]

def map_age(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if 18 <= x <= 24: return "18-24"
    elif 25 <= x <= 29: return "25-29"
    elif 30 <= x <= 34: return "30-34"
    elif 35 <= x <= 39: return "35-39"
    elif 40 <= x <= 44: return "40-44"
    elif 45 <= x <= 49: return "45-49"
    elif 50 <= x <= 54: return "50-54"
    elif 55 <= x <= 59: return "55-59"
    elif 60 <= x <= 64: return "60-64"
    elif 65 <= x <= 69: return "65-69"
    elif 70 <= x <= 74: return "70-74"
    elif 75 <= x <= 79: return "75-79"
    elif x >= 80: return "80+"
    return np.nan

def map_sex(x):
    return {1: "Male", 2: "Female"}.get(x, np.nan)

def map_edu(x):
    if pd.isna(x): return np.nan
    x = int(x)
    if x <= 15:
        return "Did not graduate high school"
    elif x <= 17:
        return "Graduated high school"
    elif x <= 20:
        return "Attended college or technical school"
    else:
        return "Graduated college or technical school"

def map_income(x):
    if pd.isna(x):
        return np.nan
    x = float(x)
    if x < 15000: return "<15k"
    elif x < 25000: return "15k-25k"
    elif x < 35000: return "25k-35k"
    elif x < 50000: return "35k-50k"
    elif x < 100000: return "50k-100k"
    elif x < 200000: return "100k-200k"
    else: return "200k+"

def map_race(row):
    # adjust this if your codebook shows a different HISP coding
    if row["HISP"] != 1:
        return "Hispanic"
    rac = row["RAC1P"]
    if rac == 1:
        return "NH-White"
    elif rac == 2:
        return "NH-Black"
    elif rac == 3:
        return "AIAN"
    elif rac == 5:
        return "Asian"
    elif rac == 6:
        return "NHOPI"
    elif rac == 7:
        return "Other"
    else:
        return "Multiracial"

def process_chunk(chunk):
    chunk = chunk.copy()
    chunk["age"] = chunk["AGEP"].apply(map_age)
    chunk["sex"] = chunk["SEX"].apply(map_sex)
    chunk["education"] = chunk["SCHL"].apply(map_edu)
    chunk["income"] = chunk["PINCP"].apply(map_income)
    chunk["race"] = chunk.apply(map_race, axis=1)

    chunk = chunk.dropna(subset=["age", "sex", "education", "income", "race", "PWGTP"])

    grouped = (
        chunk.groupby(["age", "sex", "race", "education", "income"], observed=True)["PWGTP"]
        .sum()
        .reset_index()
        .rename(columns={"PWGTP": "weight"})
    )
    return grouped

chunks = []
for chunk in pd.read_csv(file_path, usecols=usecols, chunksize=100000, low_memory=False):
    chunks.append(process_chunk(chunk))

joint = pd.concat(chunks, ignore_index=True)
joint = joint.groupby(["age", "sex", "race", "education", "income"], as_index=False)["weight"].sum()
joint["prop"] = joint["weight"] / joint["weight"].sum()

joint.to_csv("pums_joint_2020.csv", index=False)
print(joint.head())

     age     sex  race                             education     income  \
0  18-24  Female  AIAN  Attended college or technical school  100k-200k   
1  18-24  Female  AIAN  Attended college or technical school    15k-25k   
2  18-24  Female  AIAN  Attended college or technical school      200k+   
3  18-24  Female  AIAN  Attended college or technical school    25k-35k   
4  18-24  Female  AIAN  Attended college or technical school    35k-50k   

   weight          prop  
0      51  6.595600e-07  
1    1506  1.947642e-05  
2       4  5.173020e-08  
3     553  7.151700e-06  
4     290  3.750439e-06  


In [11]:
# 1) total shares sum to 1
joint["prop"].sum()

# 2) how many combinations you actually got
joint.shape[0]

# 3) no missing categories inside each variable
joint["age"].unique()
joint["sex"].unique()
joint["race"].unique()
joint["education"].unique()
joint["income"].unique()

array(['100k-200k', '15k-25k', '200k+', '25k-35k', '35k-50k', '50k-100k',
       '<15k'], dtype=object)

In [13]:
joint.groupby(["age","sex","race","education","income"])["weight"].sum().reset_index().head()

,age,sex,race,education,income,weight
0,18-24,Female,AIAN,Attended college or technical school,100k-200k,51
1,18-24,Female,AIAN,Attended college or technical school,15k-25k,1506
2,18-24,Female,AIAN,Attended college or technical school,200k+,4
3,18-24,Female,AIAN,Attended college or technical school,25k-35k,553
4,18-24,Female,AIAN,Attended college or technical school,35k-50k,290


In [17]:
import pandas as pd
import numpy as np

GROUP_COLS = ["age", "sex", "race", "education", "income"]
USECOLS = ["PWGTP", "AGEP", "SEX", "HISP", "RAC1P", "SCHL", "PINCP"]

def map_age(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if 18 <= x <= 24: return "18-24"
    elif 25 <= x <= 29: return "25-29"
    elif 30 <= x <= 34: return "30-34"
    elif 35 <= x <= 39: return "35-39"
    elif 40 <= x <= 44: return "40-44"
    elif 45 <= x <= 49: return "45-49"
    elif 50 <= x <= 54: return "50-54"
    elif 55 <= x <= 59: return "55-59"
    elif 60 <= x <= 64: return "60-64"
    elif 65 <= x <= 69: return "65-69"
    elif 70 <= x <= 74: return "70-74"
    elif 75 <= x <= 79: return "75-79"
    elif x >= 80: return "80+"
    return np.nan

def map_sex(x):
    return {1: "Male", 2: "Female"}.get(x, np.nan)

def map_edu(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x <= 15:
        return "Did not graduate high school"
    elif x <= 17:
        return "Graduated high school"
    elif x <= 20:
        return "Attended college or technical school"
    else:
        return "Graduated college or technical school"

def map_income(x):
    if pd.isna(x):
        return np.nan
    x = float(x)
    if x < 0:
        return "<15k"
    elif x < 15000:
        return "<15k"
    elif x < 25000:
        return "15k-25k"
    elif x < 35000:
        return "25k-35k"
    elif x < 50000:
        return "35k-50k"
    elif x < 100000:
        return "50k-100k"
    elif x < 200000:
        return "100k-200k"
    else:
        return "200k+"

def map_race(row):
    hisp = row["HISP"]
    rac = row["RAC1P"]

    if pd.isna(hisp) or pd.isna(rac):
        return np.nan

    # Hispanic overrides race
    if int(hisp) != 1:
        return "Hispanic"

    rac = int(rac)

    if rac == 1:
        return "NH-White"
    elif rac == 2:
        return "NH-Black"
    elif rac == 3:
        return "AIAN"
    elif rac in [4, 5, 6, 7, 8, 9, 10]:
        return "Asian"
    elif rac in [11, 12, 13, 14]:
        return "NHOPI"
    elif rac == 15:
        return "Other"
    else:
        return "Multiracial"

def process_chunk(chunk):
    chunk = chunk.copy()

    chunk["weight"] = pd.to_numeric(chunk["PWGTP"], errors="coerce")
    chunk["age"] = chunk["AGEP"].apply(map_age)
    chunk["sex"] = chunk["SEX"].apply(map_sex)
    chunk["race"] = chunk.apply(map_race, axis=1)
    chunk["education"] = chunk["SCHL"].apply(map_edu)
    chunk["income"] = chunk["PINCP"].apply(map_income)

    chunk = chunk.dropna(subset=GROUP_COLS + ["weight"])

    grouped = (
        chunk.groupby(GROUP_COLS, observed=True)["weight"]
        .sum()
        .reset_index()
        .rename(columns={"weight": "weight"})
    )
    return grouped

def process_pums_files(file_paths, year, chunksize=100000):
    partials = []

    for fp in file_paths:
        for chunk in pd.read_csv(
            fp,
            usecols=USECOLS,
            chunksize=chunksize,
            low_memory=False
        ):
            partials.append(process_chunk(chunk))

    joint = pd.concat(partials, ignore_index=True)
    joint = joint.groupby(GROUP_COLS, as_index=False)["weight"].sum()
    joint["year"] = year
    joint["prop"] = joint["weight"] / joint["weight"].sum()

    return joint

files_2020 = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusa.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusb.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusc.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusd.csv",
]

joint_2020 = process_pums_files(files_2020, 2020)
joint_2020 = joint_2020.sort_values(GROUP_COLS).reset_index(drop=True)

joint_2020.to_csv("pums_joint_2020.csv", index=False)

print(joint_2020.head())
print(joint_2020["prop"].sum())
print(joint_2020.shape)

     age     sex  race                             education     income  \
0  18-24  Female  AIAN  Attended college or technical school  100k-200k   
1  18-24  Female  AIAN  Attended college or technical school    15k-25k   
2  18-24  Female  AIAN  Attended college or technical school      200k+   
3  18-24  Female  AIAN  Attended college or technical school    25k-35k   
4  18-24  Female  AIAN  Attended college or technical school    35k-50k   

   weight  year          prop  
0      57  2020  2.250327e-07  
1    5894  2020  2.326917e-05  
2      43  2020  1.697615e-07  
3    2318  2020  9.151329e-06  
4     885  2020  3.493929e-06  
1.0
(3622, 8)


In [1]:
import pandas as pd
import numpy as np

GROUP_COLS = ["age", "sex", "race", "education", "income"]
USECOLS = ["PWGTP", "AGEP", "SEX", "HISP", "RAC1P", "SCHL", "PINCP"]

def map_age(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if 18 <= x <= 24: return "18-24"
    elif 25 <= x <= 29: return "25-29"
    elif 30 <= x <= 34: return "30-34"
    elif 35 <= x <= 39: return "35-39"
    elif 40 <= x <= 44: return "40-44"
    elif 45 <= x <= 49: return "45-49"
    elif 50 <= x <= 54: return "50-54"
    elif 55 <= x <= 59: return "55-59"
    elif 60 <= x <= 64: return "60-64"
    elif 65 <= x <= 69: return "65-69"
    elif 70 <= x <= 74: return "70-74"
    elif 75 <= x <= 79: return "75-79"
    elif x >= 80: return "80+"
    return np.nan

def map_sex(x):
    return {1: "Male", 2: "Female"}.get(x, np.nan)

def map_edu(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x <= 15:
        return "Did not graduate high school"
    elif x <= 17:
        return "Graduated high school"
    elif x <= 20:
        return "Attended college or technical school"
    else:
        return "Graduated college or technical school"

def map_income(x):
    if pd.isna(x):
        return np.nan
    x = float(x)
    if x < 0:
        return "<15k"
    elif x < 15000:
        return "<15k"
    elif x < 25000:
        return "15k-25k"
    elif x < 35000:
        return "25k-35k"
    elif x < 50000:
        return "35k-50k"
    elif x < 100000:
        return "50k-100k"
    elif x < 200000:
        return "100k-200k"
    else:
        return "200k+"

def map_race(row):
    hisp = row["HISP"]
    rac = row["RAC1P"]

    if pd.isna(hisp) or pd.isna(rac):
        return np.nan

    # Hispanic overrides race
    if int(hisp) != 1:
        return "Hispanic"

    rac = int(rac)

    if rac == 1:
        return "NH-White"
    elif rac == 2:
        return "NH-Black"
    elif rac == 3:
        return "AIAN"
    elif rac in [4, 5, 6, 7, 8, 9, 10]:
        return "Asian"
    elif rac in [11, 12, 13, 14]:
        return "NHOPI"
    elif rac == 15:
        return "Other"
    else:
        return "Multiracial"

def process_chunk(chunk):
    chunk = chunk.copy()

    chunk["weight"] = pd.to_numeric(chunk["PWGTP"], errors="coerce")
    chunk["age"] = chunk["AGEP"].apply(map_age)
    chunk["sex"] = chunk["SEX"].apply(map_sex)
    chunk["race"] = chunk.apply(map_race, axis=1)
    chunk["education"] = chunk["SCHL"].apply(map_edu)
    chunk["income"] = chunk["PINCP"].apply(map_income)

    chunk = chunk.dropna(subset=GROUP_COLS + ["weight"])

    grouped = (
        chunk.groupby(GROUP_COLS, observed=True)["weight"]
        .sum()
        .reset_index()
        .rename(columns={"weight": "weight"})
    )
    return grouped

def process_pums_files(file_paths, year, chunksize=100000):
    partials = []

    for fp in file_paths:
        for chunk in pd.read_csv(
            fp,
            usecols=USECOLS,
            chunksize=chunksize,
            low_memory=False
        ):
            partials.append(process_chunk(chunk))

    joint = pd.concat(partials, ignore_index=True)
    joint = joint.groupby(GROUP_COLS, as_index=False)["weight"].sum()
    joint["year"] = year
    joint["prop"] = joint["weight"] / joint["weight"].sum()

    return joint

files_2021 = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusa_2021.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusb_2021.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusc_2021.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusd_2021.csv",
]

joint_2021 = process_pums_files(files_2021, 2021)
joint_2021 = joint_2021.sort_values(GROUP_COLS).reset_index(drop=True)

joint_2021.to_csv("pums_joint_2021.csv", index=False)

print(joint_2021.head())
print(joint_2021["prop"].sum())
print(joint_2021.shape)

     age     sex  race                             education     income  \
0  18-24  Female  AIAN  Attended college or technical school  100k-200k   
1  18-24  Female  AIAN  Attended college or technical school    15k-25k   
2  18-24  Female  AIAN  Attended college or technical school      200k+   
3  18-24  Female  AIAN  Attended college or technical school    25k-35k   
4  18-24  Female  AIAN  Attended college or technical school    35k-50k   

   weight  year          prop  
0      55  2021  2.152538e-07  
1    5224  2021  2.044520e-05  
2      30  2021  1.174112e-07  
3    2426  2021  9.494650e-06  
4     805  2021  3.150533e-06  
1.0
(3620, 8)


In [3]:
import pandas as pd
import numpy as np

GROUP_COLS = ["age", "sex", "race", "education", "income"]
USECOLS = ["PWGTP", "AGEP", "SEX", "HISP", "RAC1P", "SCHL", "PINCP"]

def map_age(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if 18 <= x <= 24: return "18-24"
    elif 25 <= x <= 29: return "25-29"
    elif 30 <= x <= 34: return "30-34"
    elif 35 <= x <= 39: return "35-39"
    elif 40 <= x <= 44: return "40-44"
    elif 45 <= x <= 49: return "45-49"
    elif 50 <= x <= 54: return "50-54"
    elif 55 <= x <= 59: return "55-59"
    elif 60 <= x <= 64: return "60-64"
    elif 65 <= x <= 69: return "65-69"
    elif 70 <= x <= 74: return "70-74"
    elif 75 <= x <= 79: return "75-79"
    elif x >= 80: return "80+"
    return np.nan

def map_sex(x):
    return {1: "Male", 2: "Female"}.get(x, np.nan)

def map_edu(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x <= 15:
        return "Did not graduate high school"
    elif x <= 17:
        return "Graduated high school"
    elif x <= 20:
        return "Attended college or technical school"
    else:
        return "Graduated college or technical school"

def map_income(x):
    if pd.isna(x):
        return np.nan
    x = float(x)
    if x < 0:
        return "<15k"
    elif x < 15000:
        return "<15k"
    elif x < 25000:
        return "15k-25k"
    elif x < 35000:
        return "25k-35k"
    elif x < 50000:
        return "35k-50k"
    elif x < 100000:
        return "50k-100k"
    elif x < 200000:
        return "100k-200k"
    else:
        return "200k+"

def map_race(row):
    hisp = row["HISP"]
    rac = row["RAC1P"]

    if pd.isna(hisp) or pd.isna(rac):
        return np.nan

    # Hispanic overrides race
    if int(hisp) != 1:
        return "Hispanic"

    rac = int(rac)

    if rac == 1:
        return "NH-White"
    elif rac == 2:
        return "NH-Black"
    elif rac == 3:
        return "AIAN"
    elif rac in [4, 5, 6, 7, 8, 9, 10]:
        return "Asian"
    elif rac in [11, 12, 13, 14]:
        return "NHOPI"
    elif rac == 15:
        return "Other"
    else:
        return "Multiracial"

def process_chunk(chunk):
    chunk = chunk.copy()

    chunk["weight"] = pd.to_numeric(chunk["PWGTP"], errors="coerce")
    chunk["age"] = chunk["AGEP"].apply(map_age)
    chunk["sex"] = chunk["SEX"].apply(map_sex)
    chunk["race"] = chunk.apply(map_race, axis=1)
    chunk["education"] = chunk["SCHL"].apply(map_edu)
    chunk["income"] = chunk["PINCP"].apply(map_income)

    chunk = chunk.dropna(subset=GROUP_COLS + ["weight"])

    grouped = (
        chunk.groupby(GROUP_COLS, observed=True)["weight"]
        .sum()
        .reset_index()
        .rename(columns={"weight": "weight"})
    )
    return grouped

def process_pums_files(file_paths, year, chunksize=100000):
    partials = []

    for fp in file_paths:
        for chunk in pd.read_csv(
            fp,
            usecols=USECOLS,
            chunksize=chunksize,
            low_memory=False
        ):
            partials.append(process_chunk(chunk))

    joint = pd.concat(partials, ignore_index=True)
    joint = joint.groupby(GROUP_COLS, as_index=False)["weight"].sum()
    joint["year"] = year
    joint["prop"] = joint["weight"] / joint["weight"].sum()

    return joint

files_2022 = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusa_2022.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusb_2022.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusc_2022.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusd_2022.csv",
]

joint_2022 = process_pums_files(files_2022, 2022)
joint_2022 = joint_2022.sort_values(GROUP_COLS).reset_index(drop=True)

joint_2022.to_csv("pums_joint_2022.csv", index=False)

print(joint_2022.head())
print(joint_2022["prop"].sum())
print(joint_2022.shape)

     age     sex  race                             education     income  \
0  18-24  Female  AIAN  Attended college or technical school  100k-200k   
1  18-24  Female  AIAN  Attended college or technical school    15k-25k   
2  18-24  Female  AIAN  Attended college or technical school      200k+   
3  18-24  Female  AIAN  Attended college or technical school    25k-35k   
4  18-24  Female  AIAN  Attended college or technical school    35k-50k   

   weight  year          prop  
0      49  2022  1.899933e-07  
1    4689  2022  1.818119e-05  
2      26  2022  1.008128e-07  
3    2497  2022  9.681903e-06  
4     972  2022  3.768846e-06  
1.0
(3620, 8)


In [7]:
import pandas as pd
import numpy as np

GROUP_COLS = ["age", "sex", "race", "education", "income"]
USECOLS = ["PWGTP", "AGEP", "SEX", "HISP", "RAC1P", "SCHL", "PINCP"]

def map_age(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if 18 <= x <= 24: return "18-24"
    elif 25 <= x <= 29: return "25-29"
    elif 30 <= x <= 34: return "30-34"
    elif 35 <= x <= 39: return "35-39"
    elif 40 <= x <= 44: return "40-44"
    elif 45 <= x <= 49: return "45-49"
    elif 50 <= x <= 54: return "50-54"
    elif 55 <= x <= 59: return "55-59"
    elif 60 <= x <= 64: return "60-64"
    elif 65 <= x <= 69: return "65-69"
    elif 70 <= x <= 74: return "70-74"
    elif 75 <= x <= 79: return "75-79"
    elif x >= 80: return "80+"
    return np.nan

def map_sex(x):
    return {1: "Male", 2: "Female"}.get(x, np.nan)

def map_edu(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x <= 15:
        return "Did not graduate high school"
    elif x <= 17:
        return "Graduated high school"
    elif x <= 20:
        return "Attended college or technical school"
    else:
        return "Graduated college or technical school"

def map_income(x):
    if pd.isna(x):
        return np.nan
    x = float(x)
    if x < 0:
        return "<15k"
    elif x < 15000:
        return "<15k"
    elif x < 25000:
        return "15k-25k"
    elif x < 35000:
        return "25k-35k"
    elif x < 50000:
        return "35k-50k"
    elif x < 100000:
        return "50k-100k"
    elif x < 200000:
        return "100k-200k"
    else:
        return "200k+"

def map_race(row):
    hisp = row["HISP"]
    rac = row["RAC1P"]

    if pd.isna(hisp) or pd.isna(rac):
        return np.nan

    # Hispanic overrides race
    if int(hisp) != 1:
        return "Hispanic"

    rac = int(rac)

    if rac == 1:
        return "NH-White"
    elif rac == 2:
        return "NH-Black"
    elif rac == 3:
        return "AIAN"
    elif rac in [4, 5, 6, 7, 8, 9, 10]:
        return "Asian"
    elif rac in [11, 12, 13, 14]:
        return "NHOPI"
    elif rac == 15:
        return "Other"
    else:
        return "Multiracial"

def process_chunk(chunk):
    chunk = chunk.copy()

    chunk["weight"] = pd.to_numeric(chunk["PWGTP"], errors="coerce")
    chunk["age"] = chunk["AGEP"].apply(map_age)
    chunk["sex"] = chunk["SEX"].apply(map_sex)
    chunk["race"] = chunk.apply(map_race, axis=1)
    chunk["education"] = chunk["SCHL"].apply(map_edu)
    chunk["income"] = chunk["PINCP"].apply(map_income)

    chunk = chunk.dropna(subset=GROUP_COLS + ["weight"])

    grouped = (
        chunk.groupby(GROUP_COLS, observed=True)["weight"]
        .sum()
        .reset_index()
        .rename(columns={"weight": "weight"})
    )
    return grouped

def process_pums_files(file_paths, year, chunksize=100000):
    partials = []

    for fp in file_paths:
        for chunk in pd.read_csv(
            fp,
            usecols=USECOLS,
            chunksize=chunksize,
            low_memory=False
        ):
            partials.append(process_chunk(chunk))

    joint = pd.concat(partials, ignore_index=True)
    joint = joint.groupby(GROUP_COLS, as_index=False)["weight"].sum()
    joint["year"] = year
    joint["prop"] = joint["weight"] / joint["weight"].sum()

    return joint

files_2023 = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusa_2023.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusb_2023.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusc_2023.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusd_2023.csv",
]

joint_2023 = process_pums_files(files_2023, 2023)
joint_2023 = joint_2023.sort_values(GROUP_COLS).reset_index(drop=True)

joint_2023.to_csv("pums_joint_2023.csv", index=False)

print(joint_2023.head())
print(joint_2023["prop"].sum())
print(joint_2023.shape)

     age     sex  race                             education     income  \
0  18-24  Female  AIAN  Attended college or technical school  100k-200k   
1  18-24  Female  AIAN  Attended college or technical school    15k-25k   
2  18-24  Female  AIAN  Attended college or technical school      200k+   
3  18-24  Female  AIAN  Attended college or technical school    25k-35k   
4  18-24  Female  AIAN  Attended college or technical school    35k-50k   

   weight  year          prop  
0      17  2023  6.569970e-08  
1    4102  2023  1.585295e-05  
2      14  2023  5.410564e-08  
3    2661  2023  1.028394e-05  
4    1301  2023  5.027960e-06  
1.0
(3622, 8)


In [9]:
import pandas as pd
import numpy as np

GROUP_COLS = ["age", "sex", "race", "education", "income"]
USECOLS = ["PWGTP", "AGEP", "SEX", "HISP", "RAC1P", "SCHL", "PINCP"]

def map_age(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if 18 <= x <= 24: return "18-24"
    elif 25 <= x <= 29: return "25-29"
    elif 30 <= x <= 34: return "30-34"
    elif 35 <= x <= 39: return "35-39"
    elif 40 <= x <= 44: return "40-44"
    elif 45 <= x <= 49: return "45-49"
    elif 50 <= x <= 54: return "50-54"
    elif 55 <= x <= 59: return "55-59"
    elif 60 <= x <= 64: return "60-64"
    elif 65 <= x <= 69: return "65-69"
    elif 70 <= x <= 74: return "70-74"
    elif 75 <= x <= 79: return "75-79"
    elif x >= 80: return "80+"
    return np.nan

def map_sex(x):
    return {1: "Male", 2: "Female"}.get(x, np.nan)

def map_edu(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x <= 15:
        return "Did not graduate high school"
    elif x <= 17:
        return "Graduated high school"
    elif x <= 20:
        return "Attended college or technical school"
    else:
        return "Graduated college or technical school"

def map_income(x):
    if pd.isna(x):
        return np.nan
    x = float(x)
    if x < 0:
        return "<15k"
    elif x < 15000:
        return "<15k"
    elif x < 25000:
        return "15k-25k"
    elif x < 35000:
        return "25k-35k"
    elif x < 50000:
        return "35k-50k"
    elif x < 100000:
        return "50k-100k"
    elif x < 200000:
        return "100k-200k"
    else:
        return "200k+"

def map_race(row):
    hisp = row["HISP"]
    rac = row["RAC1P"]

    if pd.isna(hisp) or pd.isna(rac):
        return np.nan

    # Hispanic overrides race
    if int(hisp) != 1:
        return "Hispanic"

    rac = int(rac)

    if rac == 1:
        return "NH-White"
    elif rac == 2:
        return "NH-Black"
    elif rac == 3:
        return "AIAN"
    elif rac in [4, 5, 6, 7, 8, 9, 10]:
        return "Asian"
    elif rac in [11, 12, 13, 14]:
        return "NHOPI"
    elif rac == 15:
        return "Other"
    else:
        return "Multiracial"

def process_chunk(chunk):
    chunk = chunk.copy()

    chunk["weight"] = pd.to_numeric(chunk["PWGTP"], errors="coerce")
    chunk["age"] = chunk["AGEP"].apply(map_age)
    chunk["sex"] = chunk["SEX"].apply(map_sex)
    chunk["race"] = chunk.apply(map_race, axis=1)
    chunk["education"] = chunk["SCHL"].apply(map_edu)
    chunk["income"] = chunk["PINCP"].apply(map_income)

    chunk = chunk.dropna(subset=GROUP_COLS + ["weight"])

    grouped = (
        chunk.groupby(GROUP_COLS, observed=True)["weight"]
        .sum()
        .reset_index()
        .rename(columns={"weight": "weight"})
    )
    return grouped

def process_pums_files(file_paths, year, chunksize=100000):
    partials = []

    for fp in file_paths:
        for chunk in pd.read_csv(
            fp,
            usecols=USECOLS,
            chunksize=chunksize,
            low_memory=False
        ):
            partials.append(process_chunk(chunk))

    joint = pd.concat(partials, ignore_index=True)
    joint = joint.groupby(GROUP_COLS, as_index=False)["weight"].sum()
    joint["year"] = year
    joint["prop"] = joint["weight"] / joint["weight"].sum()

    return joint

files_2024 = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusa_2024.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusb_2024.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusc_2024.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusd_2024.csv",
]

joint_2024 = process_pums_files(files_2024, 2024)
joint_2024 = joint_2024.sort_values(GROUP_COLS).reset_index(drop=True)

joint_2024.to_csv("pums_joint_2024.csv", index=False)

print(joint_2024.head())
print(joint_2024["prop"].sum())
print(joint_2024.shape)

     age     sex  race                             education     income  \
0  18-24  Female  AIAN  Attended college or technical school  100k-200k   
1  18-24  Female  AIAN  Attended college or technical school    15k-25k   
2  18-24  Female  AIAN  Attended college or technical school      200k+   
3  18-24  Female  AIAN  Attended college or technical school    25k-35k   
4  18-24  Female  AIAN  Attended college or technical school    35k-50k   

   weight  year          prop  
0      23  2024  8.798383e-08  
1    3955  2024  1.512939e-05  
2       1  2024  3.825384e-09  
3    2476  2024  9.471650e-06  
4    1548  2024  5.921694e-06  
1.0
(3624, 8)
